# 04 · Launch PyTorch Lightning — Single Node

학습 로직은 [`lightning_trainer.py`](./lightning_trainer.py)의 `fit(...)`에 있습니다. 본 노트북은 driver 역할로 파라미터 구성·실행·결과 확인만 담당합니다.

| 섹션 | launcher | Trainer |
|------|----------|---------|
| 1×1  | (driver에서 직접 `fit` 호출) | `devices=1, num_nodes=1, strategy='auto'` |
| 1×N  | `TorchDistributor(local_mode=True).run(fit, ...)` | `devices=N, num_nodes=1, strategy='ddp'` |

1×N에서 Lightning의 `ddp_notebook` 대신 TorchDistributor를 쓰는 이유: 1×1 섹션이 driver의 CUDA를 이미 초기화하면 fork 기반 `ddp_notebook`이 실패합니다. TorchDistributor는 fresh subprocess를 띄우고, 그 안에서 Lightning이 미리 세팅된 `RANK`/`WORLD_SIZE`를 사용합니다.

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0, Autoscaling off.

In [ ]:
%run ./00-setup

## 학습 함수 import + SCRIPT_DIR

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

from lightning_trainer import fit

## 1×1 GPU

`Trainer(accelerator='gpu', devices=1, num_nodes=1)`. driver에서 `fit(...)`을 직접 호출. `lightning_trainer.fit` 내부에서 `RANK`가 없으면 rank 0으로 간주하고 `mlflow.start_run(run_id=...)`로 driver run에 attach.

In [ ]:
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

fit(
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-1x1"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=1,
    num_nodes=1,
    topology="1x1",
    script_dir=SCRIPT_DIR,
)

## 1×N GPU

`TorchDistributor(num_processes=N, local_mode=True).run(fit, ...)`로 child를 N개 띄우고, 각 child에서 `Trainer(devices=N, num_nodes=1, strategy='ddp')`로 학습. 1×1과 같은 `CONFIG`·같은 `DATA_DIR`.

In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor

NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_GPUS_PER_NODE,
    local_mode=True,
    use_gpu=True,
).run(
    fit,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-1xN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=NUM_GPUS_PER_NODE,
    num_nodes=1,
    topology="1xN",
    script_dir=SCRIPT_DIR,
)